# Notebook 02 - Fintech Archive Restore and Session Readiness

This notebook is designed to run in its own Colab runtime. Local `/content` state from Notebook 00/01 may not exist when Notebook 02 starts.

Notebook 02 restores or bootstraps a saved Fintech archive/session backup from Google Drive into the active `/content` runtime, then validates restored workspace and session readiness. Google Drive is persistent archive/session storage only; it is not the active app workspace.

Upstream backup-pack source layout is session-scoped:
`fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

Observed smoke-test backup-pack example (for context only, not a required threshold): `file_count=1340`, `shard_count=1`, `total_uncompressed_bytes=8249040`, `total_archive_bytes=2802236`.

Backfilled market data can be slow to move through Drive as many small Parquet files. Prefer packaged, sharded, or native archive restore flows when upstream commands support them. Notebook code must orchestrate native upstream commands and review small metadata summaries; it must not reimplement archive, restore, session persistence, ingestion, or generated artifact logic.

Do not commit generated archive/session payloads, restored workspace files, generated data, restore outputs, notebook outputs, execution counts, credentials, private paths, or personal Drive folder names. Restore support and exact flags remain upstream-confirmation-dependent until `fintech-market-ingestion` documents the current CLI surface.

## 1. Optional Colab Runtime Package Setup

Install or expose upstream apps only in a live Colab runtime. Do not run package installation during repository validation.

In [ ]:
# Manual Colab-only setup cell.
# Uncomment only when the Colab runtime needs the upstream package.
# !python -m pip install --upgrade pip
# !python -m pip install "pandas-market-calendars>=5.0"
# !python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion

print("Package setup is manual Colab-only. Keep installs out of repository validation.")

## 2. Optional Google Drive Mount

Mount Drive only in Colab when you intentionally need access to archive/session backup storage. Drive mount is manual and must not run in local repository validation.

In [ ]:
# Manual Colab-only Drive mount cell.
# from google.colab import drive
# drive.mount("/content/drive")

print("Drive mount is manual Colab-only. Active app work remains under /content.")

## 3. Configure Backup-Pack Source and Local Restore Target

Set a real Drive folder, session id, and backup id before any restore preview or live restore. These placeholders are intentionally invalid until replaced in a live runtime.

Notebook 02 consumes a session backup-pack source produced by Notebook 01 or another prior workflow while local backfill state existed. It does not create the backup source.

Expected source shape:
- `fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

In [ ]:
from pathlib import Path

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
SESSION_ID = "REPLACE_WITH_SESSION_ID"
BACKUP_ID = "REPLACE_WITH_BACKUP_ID"

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
DRIVE_PROJECT_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
DRIVE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / SESSION_ID
DRIVE_BACKUP_ROOT = DRIVE_SESSION_ROOT / "backups" / BACKUP_ID
DRIVE_BACKUP_MANIFEST = DRIVE_BACKUP_ROOT / "manifest.json"

FINTECH_ROOT = Path("/content/fintech-market-ingestion-demo")
OVERWRITE_POLICY = "refuse"  # Keep overwrite behavior explicit before live restore.
RESTORE_COMMAND_CANDIDATE = "fintech-restore-session"

if not str(FINTECH_ROOT).startswith("/content/"):
    raise ValueError(f"Active Fintech workspace must stay under /content: {FINTECH_ROOT}")

print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Drive session root:", DRIVE_SESSION_ROOT)
print("Drive backup/archive source root:", DRIVE_BACKUP_ROOT)
print("Drive backup manifest:", DRIVE_BACKUP_MANIFEST)
print("Local restore target workspace:", FINTECH_ROOT)
print("Overwrite policy:", OVERWRITE_POLICY)

## 4. Confirm Native Restore Command Availability

These checks report whether expected upstream command entry points are visible in the current runtime. `fintech-restore-session` is treated as the restore command candidate from the failed smoke-run review and must still be confirmed against current upstream help output before live use.

In [ ]:
import shutil

COMMAND_CANDIDATES = [
    "fintech-init-project",
    "fintech-restore-session",
    "fintech-save-session",
    "fintech-backup-data",
]

COMMAND_AVAILABLE = {name: shutil.which(name) is not None for name in COMMAND_CANDIDATES}
for name, available in COMMAND_AVAILABLE.items():
    print(f"{name}: {'available' if available else 'missing'}")

if not COMMAND_AVAILABLE.get(RESTORE_COMMAND_CANDIDATE, False):
    print("Restore command candidate is not visible. Confirm upstream CLI installation before live restore.")

## 5. Restore Help Surface

Run help commands before live restore. Missing local commands are acceptable in repository validation when configured as expected warnings.

In [ ]:
!fintech-restore-session --help

# If upstream exposes archive restore under a backup/archive command instead,
# confirm that help surface before replacing the restore candidate.
# !fintech-backup-data restore --help

## 6. Initialize Local Restore Workspace

/content is the active local workspace. Google Drive is only the archive/session backup source. Restore should target an initialized Fintech project/session workspace, not an arbitrary empty path. Initialization is manual Colab-only because it may create runtime directories under `/content`. Initialization must not run during repository validation.

In [ ]:
import shlex

INIT_PROJECT_COMMAND = [
    "fintech-init-project",
    "--root", str(FINTECH_ROOT),
    "--notebooks", "REPLACE_WITH_NOTEBOOKS_ROOT",
    "--with-session",
    "--session-name", "REPLACE_WITH_SESSION_NAME",
]

print("Fintech project initialization command preview:")
print(" ".join(shlex.quote(part) for part in INIT_PROJECT_COMMAND))
print("Run this in Colab before restore if the local target workspace does not exist.")
print("Confirm fintech-init-project flags against fintech-init-project --help before enabling live execution.")

In [ ]:
# Manual Colab-only local restore workspace initialization.
# Confirm fintech-init-project flags before enabling.
# import subprocess
# subprocess.run(INIT_PROJECT_COMMAND, check=True)

print("Local restore workspace initialization remains manual Colab-only.")

## 7. Validate Local Restore Workspace Structure

Validate the initialized `/content` workspace before restore. The target must exist and include the expected Fintech project/session directories before archive restore preflight runs.

In [ ]:
EXPECTED_RESTORE_TARGET_PATHS = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data",
    FINTECH_ROOT / "data" / "curated",
]

RESTORE_TARGET_READY = all(path.exists() for path in EXPECTED_RESTORE_TARGET_PATHS)

for path in EXPECTED_RESTORE_TARGET_PATHS:
    if not path.exists():
        print(f"BLOCKED: Missing restore target path: {path}")

if RESTORE_TARGET_READY:
    print("Restore target workspace ready.")
else:
    print("Restore target workspace is not ready. Initialize the local workspace before restore.")

## 8. Archive Restore Preflight

Run this non-mutating preflight after workspace initialization and before restore previews or live restore. It blocks placeholder Drive/session/backup values, confirms Drive is mounted, checks backup-pack source paths, and verifies the local target stays under `/content`.

In [ ]:
ARCHIVE_RESTORE_PREFLIGHT_READY = True
preflight_messages = []

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set DRIVE_FOLDER_NAME to an intentional Drive folder before restore actions.")

if SESSION_ID == "REPLACE_WITH_SESSION_ID":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set SESSION_ID to an intentional session id before restore actions.")

if BACKUP_ID == "REPLACE_WITH_BACKUP_ID":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set BACKUP_ID to an intentional backup id before restore actions.")

if not str(FINTECH_ROOT).startswith("/content/"):
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Local restore target must stay under /content: {FINTECH_ROOT}")

if OVERWRITE_POLICY not in {"refuse", "replace", "merge"}:
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set OVERWRITE_POLICY explicitly to refuse, replace, or merge before live restore.")

DRIVE_MYDRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_MYDRIVE_ROOT.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Google Drive is not mounted at /content/drive/MyDrive.")
elif not DRIVE_BACKUP_ROOT.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing backup-pack source path: {DRIVE_BACKUP_ROOT}")
elif not DRIVE_BACKUP_MANIFEST.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing backup-pack manifest path: {DRIVE_BACKUP_MANIFEST}")

if not COMMAND_AVAILABLE.get(RESTORE_COMMAND_CANDIDATE, False):
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Restore command candidate is missing: {RESTORE_COMMAND_CANDIDATE}")

for message in preflight_messages:
    print("BLOCKED:", message)

if ARCHIVE_RESTORE_PREFLIGHT_READY:
    print("Archive restore preflight ready.")
else:
    print("Archive restore preflight blocked. Fix blocked items before restore preview or live restore.")

## 9. Restore/Bootstrap Command Preview

Initialize the local restore target under `/content` first, then run restore dry-run or live restore. Confirm whether upstream restore expects a backup-pack directory (`--source`) or manifest path (`--manifest`) before live restore. Do not manually recreate backup directories under Drive, and do not use Drive as the active workspace.

In [ ]:
import shlex

RESTORE_DRY_RUN_COMMAND = [
    "fintech-restore-session",
    "--root", str(FINTECH_ROOT),
    "--adapter", "google-drive",
    "--source", str(DRIVE_BACKUP_ROOT),
    "--overwrite-policy", OVERWRITE_POLICY,
    "--dry-run",
]

print("Restore dry-run command preview (directory source variant):")
print(" ".join(shlex.quote(part) for part in RESTORE_DRY_RUN_COMMAND))
print("Confirm whether upstream restore expects backup-pack directory or manifest path before live restore.")
print("Backup-pack manifest candidate:", DRIVE_BACKUP_MANIFEST)

if not ARCHIVE_RESTORE_PREFLIGHT_READY:
    raise RuntimeError("Archive restore preflight is not ready. Do not run restore yet.")

print("Preflight is ready. Confirm upstream restore flags before uncommenting live execution.")
# import subprocess
# subprocess.run(RESTORE_DRY_RUN_COMMAND, check=True)

## 10. Manual Live Restore

Live restore is manual Colab-only. First initialize the local restore target under `/content`, then confirm the restore command and flags before enabling it. Use the backup-pack source path or manifest path only after upstream help output confirms which one is expected.

In [ ]:
# Manual Colab-only live restore.
# if not ARCHIVE_RESTORE_PREFLIGHT_READY:
#     raise RuntimeError("Archive restore preflight is not ready. Fix blocked items before live restore.")
#
# LIVE_RESTORE_COMMAND = [
#     "fintech-restore-session",
#     "--root", str(FINTECH_ROOT),
#     "--adapter", "google-drive",
#     "--source", str(DRIVE_BACKUP_ROOT),
#     "--overwrite-policy", OVERWRITE_POLICY,
# ]
#
# Confirm whether upstream expects "--source <backup-pack-dir>" or "--manifest <manifest-path>".
# Manifest candidate for reference:
# DRIVE_BACKUP_MANIFEST
#
# import subprocess
# subprocess.run(LIVE_RESTORE_COMMAND, check=True)

print("Live restore remains commented/manual-only until upstream restore flags are confirmed.")

## 9. Validate Restored Workspace Structure

After restore, validate expected local `/content` workspace paths. These checks are lightweight and should not print broad generated file listings.

In [ ]:
EXPECTED_RESTORED_PATHS = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data" / "curated",
]

for path in EXPECTED_RESTORED_PATHS:
    print(f"{path}: exists={path.exists()}")

## 10. Validate Restored Session Metadata

Review the presence of restored session manifests. Do not paste full manifest contents or generated payload listings into committed notebook source.

In [ ]:
import json

RESTORED_SESSION_MANIFESTS = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime if path.exists() else 0,
)

print("Restored session manifest count:", len(RESTORED_SESSION_MANIFESTS))

if RESTORED_SESSION_MANIFESTS:
    latest_manifest_path = RESTORED_SESSION_MANIFESTS[-1]
    print("Latest restored session manifest:", latest_manifest_path)
    latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))
    safe_keys = ["session_id", "session_name", "created_at"]
    print("Safe restored session summary keys:", [key for key in safe_keys if key in latest_manifest])
else:
    print("No restored session manifest found. Rerun restore after confirming the archive/session source.")

## 11. Validate Restored Curated/Backfilled Data Presence

Check whether curated data appears to exist after restore without printing broad file listings. Prefer summary counts over path dumps.

In [ ]:
CURATED_ROOT = FINTECH_ROOT / "data" / "curated"

if CURATED_ROOT.exists():
    parquet_count = sum(1 for _ in CURATED_ROOT.rglob("*.parquet"))
    print("Curated root exists:", CURATED_ROOT)
    print("Parquet file count:", parquet_count)
else:
    print("Curated root is missing:", CURATED_ROOT)

## 12. Runtime and Repository Boundaries

Keep active app work under `/content`. Keep Google Drive as archive/session storage only. Do not commit restored workspaces, archive packages, generated data, session payloads, restore outputs, notebook outputs, execution counts, credentials, private paths, or personal Drive folder names.

Notebook 02 consumes an existing archive/session backup. Archive creation, advanced archive shard/package inspection, archive transfer workflows, StratLake initialization, feature generation, strategy smoke tests, and backtest review remain deferred to Notebook 03+ or later notebooks.

## Notebook Summary

Notebook 02 restores or bootstraps a Fintech archive/session backup from Google Drive into `/content`, then checks workspace, session, and curated-data readiness. It is designed for separate Colab runtimes and does not assume Notebook 00/01 local `/content` state still exists.

Source layout reminder for restore inputs:
- `fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

Notebook 02 now has two readiness sides before restore: source-side backup-pack presence in Drive and target-side local `/content` project/session initialization. The notebook should initialize the Fintech workspace under `/content` rather than asking users to manually recreate directories.

Before committing source updates:

- Clear all outputs.
- Keep every code cell `execution_count` as `null`.
- Confirm no real session id, backup id, private local path, personal Drive folder, credential, token, or `.env` value is present.
- Confirm no generated archive/session payloads, restored workspace files, restore outputs, generated data, logs, screenshots, or copied manifests are present.
- Keep restore execution manual and native-command-first until upstream CLI flags are confirmed.